# Chapter 3: Unsupervised Learning and Preprocessing

## Overview
This notebook covers:
1. **Preprocessing and Feature Scaling**: Standardizing and normalizing feature ranges without data leakage.
2. **Dimensionality Reduction & Feature Extraction**: Principal Component Analysis (PCA) and non-linear manifold learning via t-SNE.
3. **Clustering Algorithms**: $k$-Means, Agglomerative Hierarchical Clustering, and DBSCAN.
4. **Cluster Evaluation Metrics**: Assessing performance using Adjusted Rand Index (ARI) and Silhouette Scores.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import load_breast_cancer, load_digits, make_moons
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

# Matplotlib visual settings
plt.rc("font", size=10)
plt.rc("axes", labelsize=11, titlesize=12)

## 1. Preprocessing and Feature Scaling

Scaling techniques transform numerical features to comparable ranges:
- **`StandardScaler`**: Scales features to zero mean ($\mu = 0$) and unit variance ($\sigma = 1$).
- **`MinMaxScaler`**: Rescales data linearly into a fixed range $[0, 1]$.
- **`RobustScaler`**: Uses median and Interquartile Range (IQR) to reduce sensitivity to extreme outliers.

> **Critical Rule**: Always fit scalers on the **training set only**, then apply `transform()` to both training and testing sets to prevent **data leakage**.

In [ ]:
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=42
)

# Instantiate scaler and fit ONLY on training data
scaler = MinMaxScaler()
scaler.fit(X_train)

# Transform both training and test instances
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Original Feature Min/Max Range:")
print(f"  Min: {X_train[:, 0].min():.2f} | Max: {X_train[:, 0].max():.2f}")

print("\nScaled Feature Min/Max Range (Training Set):")
print(f"  Min: {X_train_scaled[:, 0].min():.2f} | Max: {X_train_scaled[:, 0].max():.2f}")

print("\nScaled Feature Min/Max Range (Test Set - Scaled using Train Statistics):")
print(f"  Min: {X_test_scaled[:, 0].min():.2f} | Max: {X_test_scaled[:, 0].max():.2f}")

## 2. Dimensionality Reduction: Principal Component Analysis (PCA)

PCA finds directions of maximum variance in high-dimensional feature spaces and rotates data onto orthogonal axes (components):

$$\mathbf{z}_1 = w_{11}x_1 + w_{12}x_2 + \dots + w_{1p}x_p$$

PCA is primarily used for **2D/3D visualization** and **feature reduction** prior to downstream estimator fitting.

In [ ]:
# Scale data before applying PCA
scaler_std = StandardScaler()
X_cancer_scaled = scaler_std.fit_transform(cancer.data)

# Reduce 30 continuous features down to 2 principal components
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cancer_scaled)

print(f"Original Data Shape:  {X_cancer_scaled.shape}")
print(f"Reduced Data Shape:   {X_pca.shape}")
print(f"Explained Variance Ratio by Component: {pca.explained_variance_ratio_}")
print(f"Total Variance Retained: {np.sum(pca.explained_variance_ratio_) * 100:.2f}%")

# Scatter plot of top 2 principal components
plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cancer.target, cmap="viridis", alpha=0.7)
plt.xlabel("First Principal Component")
plt.ylabel("Second Principal Component")
plt.title("2D Projection of Breast Cancer Dataset via PCA")
plt.colorbar(scatter, label="Target Class (0: Malignant, 1: Benign)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Manifold Learning: t-Distributed Stochastic Neighbor Embedding (t-SNE)

Unlike PCA (which performs linear projection), **t-SNE** is a non-linear manifold learning algorithm optimized for **exploratory data visualization**. It preserves local neighborhood distances, placing similar points close together in lower dimensions.

In [ ]:
digits = load_digits()

# Apply t-SNE to reduce 64 pixel values down to 2 dimensions
tsne = TSNE(n_components=2, random_state=42, init="pca", learning_rate="auto")
digits_tsne = tsne.fit_transform(digits.data)

plt.figure(figsize=(9, 6))
plt.scatter(
    digits_tsne[:, 0],
    digits_tsne[:, 1],
    c=digits.target,
    cmap="tab10",
    s=15,
    alpha=0.8
)
plt.colorbar(label="Digit Class (0-9)")
plt.xlabel("t-SNE Feature 1")
plt.ylabel("t-SNE Feature 2")
plt.title("2D Manifold Visualization of Digits Dataset using t-SNE")
plt.tight_layout()
plt.show()

## 4. Clustering Algorithms

We evaluate three distinct partition strategies:
1. **$k$-Means**: Partitioning algorithm assigning instances to nearest cluster centroids using Euclidean distance. Assumes spherical, isotropic cluster shapes.
2. **Agglomerative Clustering**: Bottom-up hierarchical merging based on linkage criteria (e.g., Ward's minimum variance).
3. **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**: Groups points in high-density regions and flags sparse noise points as label `-1`. Handles complex non-spherical geometric structures.

In [ ]:
# Synthetic non-spherical dataset
X_moons, y_moons = make_moons(n_samples=250, noise=0.07, random_state=42)

# Fit clustering algorithms
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_moons)
agg = AgglomerativeClustering(n_clusters=2, linkage="ward").fit_predict(X_moons)
dbscan = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_moons)

# Visualize cluster boundaries
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
algorithms = [("k-Means", kmeans), ("Agglomerative", agg), ("DBSCAN", dbscan)]

for ax, (title, labels) in zip(axes, algorithms):
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap="Accent", s=40, edgecolors="k")
    ax.set_title(title)
    ax.set_xlabel("Feature 0")
    ax.set_ylabel("Feature 1")
    ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 5. Quantitative Evaluation of Clustering Performance

Measuring cluster quality requires distinct metrics depending on ground-truth label availability:
- **Adjusted Rand Index (ARI)**: Evaluates similarity between predicted cluster partitions and ground-truth target labels. Ranges from $[-1, 1]$ ($1.0 = \text{perfect match}$).
- **Silhouette Coefficient**: Measures cluster compactness vs. separation **without ground-truth labels**. Ranges from $[-1, 1]$ ($1.0 = \text{dense, well-separated clusters}$).

In [ ]:
models = {
    "k-Means": kmeans,
    "Agglomerative": agg,
    "DBSCAN": dbscan
}

metrics_list = []

for name, labels in models.items():
    # Compute ground-truth ARI score
    ari = adjusted_rand_score(y_moons, labels)

    # Compute ungrounded Silhouette Score
    sil = silhouette_score(X_moons, labels)

    metrics_list.append({
        "Algorithm": name,
        "Adjusted Rand Index (ARI)": round(ari, 4),
        "Silhouette Score": round(sil, 4)
    })

df_metrics = pd.DataFrame(metrics_list)
print("Clustering Performance Comparison on Two Moons Dataset:")
print(df_metrics.to_string(index=False))